# Detection - YOLO11x Transfer Learning on VisDrone
### YOLO11: https://github.com/ultralytics/ultralytics
### VisDrone Dataset: https://github.com/VisDrone/VisDrone-Dataset

**Transfer Learning:**
YOLO11x is pretrained on COCO — a dataset of ground-level images across 80 classes. Aerial imagery looks fundamentally different: vehicles appear as small, top-down rectangles with no side profile. Transfer learning adapts the pretrained model to this new domain by freezing the backbone (which already extracts useful low-level features) and retraining only the detection head on VisDrone aerial data.

**VisDrone 2019 DET:**
Drone-captured imagery with 10 traffic-relevant classes: pedestrian, people, bicycle, car, van, truck, tricycle, awning-tricycle, bus, motor.

**Freeze Strategy:**
The first 10 backbone layers are frozen during fine-tuning. These layers detect edges, textures, and shapes that transfer across domains. Only the deeper layers and detection head are retrained, which reduces overfitting and training time while leveraging the pretrained feature representations.

---
## Cell 1 - Environment Setup

In [ ]:
# import dependencies
import sys
import os
import shutil
import torch
import subprocess
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
from PIL import Image
from IPython.display import display as ipy_display, Image as IPyImage

# checks that dependencies are available
print(f"Python      : {sys.version}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    subprocess.run(['bash', '-c', 'module load cuda/11.8.0'], shell=False)

# File paths for this project
BASE_DIR    = os.path.expanduser('~/TrafficRouting')
IMAGES_DIR  = os.path.join(BASE_DIR, 'detection', 'images')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'detection', 'predictions')
MODELS_DIR  = os.path.join(BASE_DIR, 'detection', 'models')
WEBSITE_DIR = os.path.join(BASE_DIR, 'website', 'results')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(WEBSITE_DIR, exist_ok=True)

# Checkpoint paths used across cells
CHECKPOINT_50  = os.path.join(MODELS_DIR, 'yolo11x_visdrone_50epoch.pt')
HF_WEIGHTS_URL = 'https://huggingface.co/Bluehippo321/yolo11x-visdrone/resolve/main/yolo11x_visdrone_50epoch.pt'

print(f"\nBase        : {BASE_DIR}")
print(f"Images      : {IMAGES_DIR}")
print(f"Output      : {OUTPUT_DIR}")
print(f"Models      : {MODELS_DIR}")

---
## Cell 1B - Download Pre-trained Weights

Downloads the pre-trained YOLO11x checkpoint from HuggingFace. Skip this cell and run Cell 5A instead if you want to retrain from scratch.

**Weights:** https://huggingface.co/Bluehippo321/yolo11x-visdrone

In [ ]:
import urllib.request

if os.path.exists(CHECKPOINT_50):
    print(f"Weights already present: {CHECKPOINT_50}")
    print("Skipping download.")
else:
    print(f"Downloading pre-trained YOLO11x weights from HuggingFace...")
    print(f"Source : {HF_WEIGHTS_URL}")
    print(f"Dest   : {CHECKPOINT_50}")
    print()

    def reporthook(count, block_size, total_size):
        mb_done  = count * block_size / 1024 / 1024
        mb_total = total_size / 1024 / 1024
        print(f"\r  {mb_done:.1f} / {mb_total:.1f} MB", end='', flush=True)

    urllib.request.urlretrieve(HF_WEIGHTS_URL, CHECKPOINT_50, reporthook)
    print()
    print("Download complete.")

print()
size_mb = os.path.getsize(CHECKPOINT_50) / 1024 / 1024
print(f"Checkpoint : {CHECKPOINT_50}")
print(f"Size       : {size_mb:.1f} MB")

---
## Cell 2 - Verify Images

In [ ]:
# For each image, make sure it is the correct format and show its size

image_files = sorted([
    f for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

if not image_files:
    print(f"No images found in {IMAGES_DIR}")
    print("Add aerial images named img0.jpg, img1.jpg, etc. and re-run this cell.")
else:
    print(f"Found {len(image_files)} image(s):")
    for img in image_files:
        path = os.path.join(IMAGES_DIR, img)
        im = Image.open(path)
        print(f"  {img}  ({im.width}x{im.height} px)")

---
## Cell 3 - BEFORE Transfer Learning - Base YOLO11x (COCO)

Running the base YOLO11x model pretrained on COCO on aerial images. COCO contains ground-level imagery — vehicles are photographed from the side. Aerial vehicles appear as small top-down rectangles which COCO features do not represent well, so we expect weak and often incorrect detections.

In [ ]:
from ultralytics import YOLO

print("Loading base YOLO11x (COCO pretrained)...")
base_model = YOLO('yolo11x.pt')
print(f"Base model loaded - {len(base_model.names)} COCO classes")

print()
print("=" * 55)
print(" BEFORE Transfer Learning - COCO base model on aerial images")
print("=" * 55)

for img_file in image_files:
    img_path = os.path.join(IMAGES_DIR, img_file)

    results = base_model.predict(
        source=img_path,
        conf=0.20,
        save=False,
        verbose=False
    )
    result = results[0]
    n_det  = len(result.boxes)

    print(f"\n{img_file} - {n_det} detection(s):")

    fig, ax = plt.subplots(1, figsize=(12, 8))
    img_arr = mpimg.imread(img_path)
    ax.imshow(img_arr)

    for box in result.boxes:
        x1, y1, x2, y2 = [v.item() for v in box.xyxy[0]]
        conf  = box.conf[0].item()
        cls   = int(box.cls[0].item())
        label = base_model.names[cls]
        print(f"  {label:<22} {conf:.3f}")

        rect = patches.Rectangle(
            (x1, y1), x2 - x1, y2 - y1,
            linewidth=2, edgecolor='#e74c3c', facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f"{label} {conf:.2f}", color='white', fontsize=7,
                bbox=dict(facecolor='#e74c3c', alpha=0.85, pad=1, edgecolor='none'))

    stem = os.path.splitext(img_file)[0]
    ax.set_title(f'BEFORE Fine-tuning (COCO Base Model) — {img_file}',
                 fontsize=12, fontweight='bold')
    ax.axis('off')

    out_path = os.path.join(OUTPUT_DIR, f"before_{stem}.png")
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close()
    ipy_display(IPyImage(filename=out_path))
    print(f"  → Saved: {out_path}")

print()
print("BEFORE inference complete.")

---
## Cell 4 - Verify VisDrone Dataset

VisDrone 2019 DET contains drone-captured imagery specifically for aerial object detection. The dataset is built into ultralytics and will auto-download if not already present.

In [ ]:
import ultralytics

# VisDrone.yaml is bundled with ultralytics
yaml_path = os.path.join(
    os.path.dirname(ultralytics.__file__),
    'cfg', 'datasets', 'VisDrone.yaml'
)

visdrone_root = os.path.expanduser('~/TrafficRouting/datasets/VisDrone')
train_dir     = os.path.join(visdrone_root, 'VisDrone2019-DET-train', 'images')
val_dir       = os.path.join(visdrone_root, 'VisDrone2019-DET-val',   'images')

print(f"VisDrone YAML  : {yaml_path}")
print(f"Dataset root   : {visdrone_root}")
print()

if os.path.exists(train_dir) and os.path.exists(val_dir):
    n_train = len([f for f in os.listdir(train_dir) if f.lower().endswith('.jpg')])
    n_val   = len([f for f in os.listdir(val_dir)   if f.lower().endswith('.jpg')])
    print(f"Status         : Found")
    print(f"Train images   : {n_train}")
    print(f"Val images     : {n_val}")
else:
    print("Dataset not found locally. Triggering download (~1.5 GB)...")
    print("This may take several minutes.")
    print()
    from ultralytics.data.utils import check_det_dataset
    check_det_dataset('VisDrone.yaml')
    print("Download complete.")

# VisDrone class names
VISDRONE_CLASSES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van',
    'truck', 'tricycle', 'awning-tricycle', 'bus', 'motor'
]
print()
print(f"VisDrone classes ({len(VISDRONE_CLASSES)}):")
for i, cls in enumerate(VISDRONE_CLASSES):
    print(f"  {i}: {cls}")

---
## Cell 5A - Fine-tune YOLO11x on VisDrone - 50 Epochs (Default)

Fine-tuning YOLO11x on VisDrone with `freeze=10` — the first 10 backbone layers are frozen, preserving their pretrained low-level feature representations. Only the deeper layers and detection head are updated, adapting the model from COCO ground-level imagery to VisDrone aerial imagery.

**Note:** Pre-trained weights are available on HuggingFace — run Cell 1B to download them instead of retraining. This cell is provided to show the full training pipeline and allow replication.

In [ ]:
from ultralytics import YOLO

EPOCHS_DEFAULT = 50
RUN_NAME_50    = 'finetune_50epoch'

print(f"Starting YOLO11n fine-tuning on VisDrone")
print(f"  Epochs      : {EPOCHS_DEFAULT}")
print(f"  Freeze      : 10 backbone layers")
print(f"  Image size  : 640")
print(f"  Batch       : 8")
print(f"  Device      : {device}")
print()

model_50 = YOLO('yolo11x.pt')

model_50.train(
    data='VisDrone.yaml',
    epochs=EPOCHS_DEFAULT,
    imgsz=640,
    batch=8,
    freeze=10,
    device=0,
    project=OUTPUT_DIR,
    name=RUN_NAME_50,
    exist_ok=True,
    plots=True,
    verbose=True
)

# Copy best checkpoint to models dir
best_pt = os.path.join(OUTPUT_DIR, RUN_NAME_50, 'weights', 'best.pt')
shutil.copy(best_pt, CHECKPOINT_50)
shutil.copy(best_pt, os.path.join(WEBSITE_DIR, 'yolo11n_visdrone_50epoch.pt'))

print()
print(f"Training complete.")
print(f"Best checkpoint : {CHECKPOINT_50}")

# Validate to get clean final metrics
metrics_50 = model_50.val(data='VisDrone.yaml', device=0, verbose=False, split='val')
print(f"mAP50           : {metrics_50.box.map50:.4f}")
print(f"mAP50-95        : {metrics_50.box.map:.4f}")

# Display training curves (saved automatically by ultralytics)
curves_path = os.path.join(OUTPUT_DIR, RUN_NAME_50, 'results.png')
if os.path.exists(curves_path):
    shutil.copy(curves_path, os.path.join(WEBSITE_DIR, 'training_curves_50epoch.png'))
    ipy_display(IPyImage(filename=curves_path))

---
## Cell 5B - Fine-tune YOLO11n on VisDrone - Custom Epochs

In [ ]:
from ultralytics import YOLO

epochs_input   = input("Enter number of epochs for custom run (e.g. 10, 25): ").strip()
EPOCHS_CUSTOM  = int(epochs_input)
RUN_NAME_CUST  = f'finetune_{EPOCHS_CUSTOM}epoch'
CHECKPOINT_CUST = os.path.join(MODELS_DIR, f'yolo11n_visdrone_{EPOCHS_CUSTOM}epoch.pt')

print()
print(f"Starting custom fine-tuning run")
print(f"  Epochs      : {EPOCHS_CUSTOM}")
print(f"  Freeze      : 10 backbone layers")
print()

model_cust = YOLO('yolo11x.pt')

model_cust.train(
    data='VisDrone.yaml',
    epochs=EPOCHS_CUSTOM,
    imgsz=640,
    batch=8,
    freeze=10,
    device=0,
    project=OUTPUT_DIR,
    name=RUN_NAME_CUST,
    exist_ok=True,
    plots=True,
    verbose=True
)

best_pt_cust = os.path.join(OUTPUT_DIR, RUN_NAME_CUST, 'weights', 'best.pt')
shutil.copy(best_pt_cust, CHECKPOINT_CUST)

metrics_cust = model_cust.val(data='VisDrone.yaml', device=0, verbose=False, split='val')

print()
print(f"Custom training complete.")
print(f"Checkpoint  : {CHECKPOINT_CUST}")
print(f"mAP50       : {metrics_cust.box.map50:.4f}")
print(f"mAP50-95    : {metrics_cust.box.map:.4f}")

curves_path_cust = os.path.join(OUTPUT_DIR, RUN_NAME_CUST, 'results.png')
if os.path.exists(curves_path_cust):
    ipy_display(IPyImage(filename=curves_path_cust))

---
## Cell 6 - Evaluate - mAP Comparison Before vs After

The base COCO model is evaluated on VisDrone's validation set. Since COCO classes do not align with VisDrone classes, the base model scores near zero — this is the expected baseline. The fine-tuned model is then evaluated on the same val set to show the improvement from transfer learning.

In [ ]:
from ultralytics import YOLO

print("Evaluating models on VisDrone validation set...")
print()

# Base model — COCO classes vs VisDrone classes, expected near-zero mAP
print("[1/2] Base YOLO11n (COCO)...")
base_eval   = YOLO('yolo11x.pt')
base_metrics = base_eval.val(
    data='VisDrone.yaml', device=0, verbose=False, split='val'
)
base_map50   = base_metrics.box.map50
base_map5095 = base_metrics.box.map
print(f"  mAP50      : {base_map50:.4f}")
print(f"  mAP50-95   : {base_map5095:.4f}")

# Fine-tuned 50 epoch
print()
print("[2/2] Fine-tuned YOLO11x (50 epoch, VisDrone)...")
ft_eval    = YOLO(CHECKPOINT_50)
ft_metrics = ft_eval.val(
    data='VisDrone.yaml', device=0, verbose=False, split='val'
)
ft_map50   = ft_metrics.box.map50
ft_map5095 = ft_metrics.box.map
print(f"  mAP50      : {ft_map50:.4f}")
print(f"  mAP50-95   : {ft_map5095:.4f}")

print()
print("=" * 50)
print(f"{'Model':<30} {'mAP50':>7} {'mAP50-95':>9}")
print("-" * 50)
print(f"{'Base YOLO11x (COCO)':<30} {base_map50:>7.4f} {base_map5095:>9.4f}")
print(f"{'Fine-tuned 50ep (VisDrone)':<30} {ft_map50:>7.4f} {ft_map5095:>9.4f}")
print("=" * 50)

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Transfer Learning — mAP Comparison on VisDrone Val Set',
             fontsize=13, fontweight='bold')

model_labels = ['Base YOLO11x\n(COCO)', 'Fine-tuned\n(50 epochs, VisDrone)']
colors       = ['#95a5a6', '#2ecc71']

for ax, metric_vals, metric_name in [
    (axes[0], [base_map50,   ft_map50],   'mAP@0.5'),
    (axes[1], [base_map5095, ft_map5095], 'mAP@0.5:0.95')
]:
    bars = ax.bar(model_labels, metric_vals, color=colors, width=0.45)
    ax.set_title(metric_name, fontsize=11)
    ax.set_ylabel('mAP Score')
    ax.set_ylim(0, max(metric_vals) * 1.25 + 0.05)
    for bar, val in zip(bars, metric_vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
map_path = os.path.join(OUTPUT_DIR, 'map_comparison.png')
plt.savefig(map_path, dpi=120, bbox_inches='tight')
plt.close()
shutil.copy(map_path, os.path.join(WEBSITE_DIR, 'map_comparison.png'))
ipy_display(IPyImage(filename=map_path))
print(f"\nSaved: {map_path}")

---
## Cell 7 - Interactive - Before vs After Comparison

Pick an image and checkpoint to run a side-by-side before/after detection comparison.

In [ ]:
from ultralytics import YOLO

img_id   = input("Enter image number to predict (e.g. 0, 1, 2...): ").strip()
img_file = f"img{img_id}.jpg"
img_path = os.path.join(IMAGES_DIR, img_file)

if not os.path.exists(img_path):
    print(f"Image not found: {img_path}")
    print(f"Make sure img{img_id}.jpg exists in {IMAGES_DIR}")
else:
    print()
    print("Which fine-tuned checkpoint to use?")
    print("  1 - 50 epoch (default)")
    print("  2 - Custom epoch run")
    choice = input("Enter 1 or 2: ").strip()

    if choice == '2':
        ep = input("Enter the epoch count of the custom run (e.g. 10, 25): ").strip()
        ckpt_path = os.path.join(MODELS_DIR, f'yolo11x_visdrone_{ep}epoch.pt')
        if not os.path.exists(ckpt_path):
            print(f"No checkpoint found for {ep} epochs, falling back to 50 epoch.")
            ckpt_path = CHECKPOINT_50
            ep = '50'
    else:
        ckpt_path = CHECKPOINT_50
        ep = '50'

    print(f"\nUsing checkpoint: {ckpt_path}")
    print(f"Running before/after comparison on {img_file}...")

    # Before: base COCO model
    base = YOLO('yolo11x.pt')
    res_before = base.predict(source=img_path, conf=0.20, save=False, verbose=False)[0]

    # After: fine-tuned model
    ft = YOLO(ckpt_path)
    res_after = ft.predict(source=img_path, conf=0.20, save=False, verbose=False)[0]

    print(f"\nBEFORE : {len(res_before.boxes)} detection(s)")
    for box in res_before.boxes:
        cls   = int(box.cls[0].item())
        conf  = box.conf[0].item()
        label = base.names[cls]
        print(f"  {label:<22} {conf:.3f}")

    print(f"\nAFTER  : {len(res_after.boxes)} detection(s)")
    for box in res_after.boxes:
        cls   = int(box.cls[0].item())
        conf  = box.conf[0].item()
        label = ft.names[cls]
        print(f"  {label:<22} {conf:.3f}")

    # Side-by-side comparison plot
    fig, axes = plt.subplots(1, 3, figsize=(22, 7))
    fig.suptitle(f'Transfer Learning — Before vs After — {img_file}',
                 fontsize=13, fontweight='bold')

    img_arr = mpimg.imread(img_path)

    # Original image
    axes[0].imshow(img_arr)
    axes[0].set_title('Input Image', fontsize=11)
    axes[0].axis('off')

    # BEFORE detections
    axes[1].imshow(img_arr)
    for box in res_before.boxes:
        x1, y1, x2, y2 = [v.item() for v in box.xyxy[0]]
        conf  = box.conf[0].item()
        label = base.names[int(box.cls[0].item())]
        rect  = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=2, edgecolor='#e74c3c', facecolor='none')
        axes[1].add_patch(rect)
        axes[1].text(x1, y1 - 4, f"{label} {conf:.2f}", color='white', fontsize=6,
                     bbox=dict(facecolor='#e74c3c', alpha=0.85, pad=1, edgecolor='none'))
    axes[1].set_title(f'BEFORE — Base COCO Model\n({len(res_before.boxes)} detections)',
                      fontsize=10)
    axes[1].axis('off')

    # AFTER detections
    axes[2].imshow(img_arr)
    for box in res_after.boxes:
        x1, y1, x2, y2 = [v.item() for v in box.xyxy[0]]
        conf  = box.conf[0].item()
        label = ft.names[int(box.cls[0].item())]
        rect  = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                   linewidth=2, edgecolor='#2ecc71', facecolor='none')
        axes[2].add_patch(rect)
        axes[2].text(x1, y1 - 4, f"{label} {conf:.2f}", color='white', fontsize=6,
                     bbox=dict(facecolor='#27ae60', alpha=0.85, pad=1, edgecolor='none'))
    axes[2].set_title(f'AFTER — Fine-tuned on VisDrone ({ep} epochs)\n({len(res_after.boxes)} detections)',
                      fontsize=10)
    axes[2].axis('off')

    plt.tight_layout()
    out_path  = os.path.join(OUTPUT_DIR, f"interactive_comparison_img{img_id}.png")
    site_path = os.path.join(WEBSITE_DIR, f"comparison_img{img_id}.png")
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.close()
    shutil.copy(out_path, site_path)
    ipy_display(IPyImage(filename=out_path))
    print(f"\nDone — {img_file} processed successfully.")